In [17]:
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [18]:
class CHIMEFRBDataset(Dataset):
    def __init__(self, hdf5_path, catalog_path, target_length):
        self.hdf5_path = hdf5_path
        self.dt = 0.009830400085775182*1000

        self.target_length = target_length
        cat = pd.read_csv(catalog_path, low_memory=False)
        cat["repeater_name"] = cat["repeater_name"].fillna("").str.strip()
        self.repeater_set = set(
            cat.loc[cat["repeater_name"] != "", "tns_name"].str.strip()
        )
        with h5py.File(hdf5_path, "r") as f:
            self.keys = list(f.keys())

        
    @staticmethod
    def _pad_or_crop(wfall, target_length, center_idx):
        n_freq, n_time = wfall.shape
        half = target_length // 2
        start = center_idx - half
        end = start + target_length
        
        if start >= 0 and end <= n_time:
            return wfall[:, start:end]         
        
        src_start = max(start, 0)
        src_end = min(end, n_time)
        out = np.zeros((n_freq, target_length), dtype=wfall.dtype)
        dst_start = src_start - start
        out[:, dst_start:dst_start + (src_end - src_start)] = wfall[:, src_start:src_end]
        return out
    
    
    def __len__(self):
        return len(self.keys)
    
    
    def __getitem__(self, idx):
        key = self.keys[idx]
        
        with h5py.File(self.hdf5_path, "r") as f:
            wfall = f[key]["wfall_plot"][:]
            extent = np.array(f[key]["extent"])
            
        wfall = wfall.astype(np.float32)
        std = wfall.std(axis=1, keepdims=True)
        std[std == 0] = 1.0          # avoid divide-by-zero for masked channels
        wfall = (wfall - wfall.mean(axis=1, keepdims=True)) / std
        

        peak = round(-extent[0] / self.dt)
        wfall = self._pad_or_crop(wfall, self.target_length, peak)
        tensor = torch.from_numpy(wfall)
        label = torch.tensor(int(key in self.repeater_set), dtype=torch.long)
        
        return tensor, label



In [19]:
def make_dataloader(
    hdf5_path: str,
    catalog_path: str,
    target_length: int,
    batch_size: int = 32,
    shuffle: bool = True,
    num_workers: int = 0,
    train_frac: float = 0.8,
    seed: int = 42,
):
    dataset = CHIMEFRBDataset(hdf5_path, catalog_path, target_length)

    n_total = len(dataset)
    n_train = int(n_total * train_frac)
    n_val = n_total - n_train

    generator = torch.Generator().manual_seed(seed)
    train_ds, val_ds = torch.utils.data.random_split(
        dataset, [n_train, n_val], generator=generator
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    # Print class balance
    labels = [dataset[i][1].item() for i in range(n_total)]
    n_rep = sum(labels)
    print(f"Dataset: {n_total} bursts | {n_rep} repeaters ({100*n_rep/n_total:.1f}%) "
          f"| {n_total-n_rep} non-repeaters")
    print(f"Train: {n_train} | Val: {n_val}")

    return train_loader, val_loader

In [20]:
TARGET_LENGTH = 128 

train_loader, val_loader = make_dataloader(
    hdf5_path="all_bursts.hdf5",
    catalog_path="chimefrbcat2.csv",
    target_length=TARGET_LENGTH,
    batch_size=32,
    num_workers=0,
)

# Sanity check
for wfall_batch, label_batch in train_loader:
    print(f"Batch shape : {wfall_batch.shape}")
    print(f"Labels      : {label_batch}")
    break


Dataset: 4536 bursts | 981 repeaters (21.6%) | 3555 non-repeaters
Train: 3628 | Val: 908
Batch shape : torch.Size([32, 256, 128])
Labels      : tensor([0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 0])


Need 

- Positional Encoding
- Masking logic
- Encoder blocks
- Decoder blocks
- CLS Token


- Forward
    - Encoder:
        - Embed input frequency snapshot
        - Add positional encoding
        - Mask input
        - Add CLS token
        - Pass through encoder blocks
        - Return x, mask, ids_restore
    - Decoder:
        - Insert masked tokens into the sequence
        - Add positional encoding
        - Pass through decoder blocks
        - Project decoder into prediction space
        - Strip CLS token
        - Return x


In [21]:
# # sin-cos positional encoding

# class SinusoidalPE(nn.Module):
#     def __init__(self, seq_len, embed_dim):
#         super().__init__()
#         pe = torch.zeros(seq_len, embed_dim)
#         pos = torch.arange(seq_len).unsqueeze(1)
#         div = torch.exp(torch.arange(0, embed_dim, 2) * (-np.log(10000) / embed_dim))
#         pe[:, 0::2] = torch.sin(pos * div)
#         pe[:, 1::2] = torch.cos(pos * div)
#         self.register_buffer("pe", pe)

#     def forward(self, x):
#         return x + self.pe  # broadcast over batch
    
    
# class FRBMaskedAutoencoder(nn.Module):
#     def __init__(self, seq_len, n_freq, embed_dim, mask_ratio=0.75):
#         super().__init__()
#         self.seq_len = seq_len
#         self.n_freq = n_freq
#         self.embed_dim = embed_dim
#         self.mask_ratio = mask_ratio
        
#         # Encoder
#         self.enc_proj = nn.Linear(n_freq, embed_dim)  # project freq channels to embed dim
#         self.enc_pe = SinusoidalPE(seq_len+1, embed_dim)
#         self.enc_blocks = nn.ModuleList([
#             nn.TransformerEncoderLayer(embed_dim, nhead=2, batch_first=True) for _ in range(2)
#         ])
        
#         self.cls_token = nn.Parameter(torch.zeros(embed_dim))
        
#         # Decoder
#         self.mask_token = nn.Parameter(torch.zeros(embed_dim))
#         self.dec_pe = SinusoidalPE(seq_len+1, embed_dim)
#         self.dec_blocks = nn.ModuleList([
#             nn.TransformerEncoderLayer(embed_dim, nhead=2, batch_first=True) for _ in range(2)
#         ])
#         self.dec_proj = nn.Linear(embed_dim, n_freq)  # project back to freq space
        
#         self.cls_head = nn.Linear(self.embed_dim, 1)
        
#         self.enc_embed = nn.Linear(embed_dim, embed_dim)
#         self.dec_embed = nn.Linear(embed_dim, embed_dim)
        
#         self.enc_norm = nn.LayerNorm(embed_dim)
#         self.dec_norm = nn.LayerNorm(embed_dim)

        
        
#     def preprocess(self, x):
#         # x: (B, 256, T)
#         x = self.enc_proj(x.permute(0, 2, 1))  # (B, T, E)
#         return x
    
#     def mask_input(self, x, mask_ratio):
        
#         N, L, D = x.shape  # batch, length, dim
#         len_keep = int(L * (1 - mask_ratio))
        
#         noise = torch.rand(N, L, device=x.device)  # noise in [0, 1]
        
#         # sort noise for each sample
#         ids_shuffle = torch.argsort(noise, dim=1)  # ascend: small is keep, large is remove
#         ids_restore = torch.argsort(ids_shuffle, dim=1)

#         # keep the first subset
#         ids_keep = ids_shuffle[:, :len_keep]
#         x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))

#         # generate the binary mask: 0 is keep, 1 is remove
#         mask = torch.ones([N, L], device=x.device)
#         mask[:, :len_keep] = 0
#         # unshuffle to get the binary mask
#         mask = torch.gather(mask, dim=1, index=ids_restore)

#         return x_masked, mask, ids_restore
    
    
#     def encoder(self, x):
#         x = self.enc_embed(x)        

#         x = x + self.enc_pe.pe[1:self.seq_len+1]                  # PE on full sequence first
#         x_masked, mask, ids_restore = self.mask_input(x, self.mask_ratio)
#         cls_token = self.cls_token + self.enc_pe.pe[0]
#         cls_tokens = cls_token.expand(x_masked.size(0), 1, -1)
#         x_masked = torch.cat([cls_tokens, x_masked], dim=1)
#         for block in self.enc_blocks:
#             x_masked = block(x_masked)
            
#         # x_masked = self.enc_norm(x_masked)
        
#         return x_masked, mask, ids_restore
    
    
#     def decoder(self, x, mask, ids_restore):
#         # embed tokens
#         x = self.dec_embed(x)

#         # append mask tokens to sequence
#         mask_tokens = self.mask_token.repeat(x.shape[0], ids_restore.shape[1] + 1 - x.shape[1], 1)
#         x_ = torch.cat([x[:, 1:, :], mask_tokens], dim=1)  # no cls token
#         x_ = torch.gather(x_, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, x.shape[2]))  # unshuffle
#         x = torch.cat([x[:, :1, :], x_], dim=1)  # append cls token

#         # add pos embed
#         x = self.dec_pe(x)

#         # apply Transformer blocks
#         for blk in self.dec_blocks:
#             x = blk(x)
            
#         # x = self.dec_norm(x)

#         # predictor projection
#         cls = x[:, 0, :]
#         x = self.dec_proj(x[:, 1:, :])
#         return x, cls
    
#     def forward(self, x):
#         x = self.preprocess(x)
#         # embed x using linear layer
#         x_masked, mask, ids_restore = self.encoder(x)
#         x_recon, cls_token = self.decoder(x_masked, mask, ids_restore)
#         cls_out = self.cls_head(cls_token)
        
#         return x_recon, cls_out.squeeze(-1), mask
    
#     def loss(self, x_recon, x_orig, cls_out, labels, mask):
#         recon_loss = ((x_recon - x_orig) ** 2).mean(dim=2)  # MSE over freq channels
#         recon_loss = (recon_loss * mask).sum() / mask.sum()
#         cls_loss = nn.BCEWithLogitsLoss()(cls_out, labels.float())
#         return recon_loss + cls_loss
    
    
# def compute_loss(cls_out, x_recon, mask, wfall, labels, alpha=1.0, beta=1.0, pos_weight=None, device='cpu'):
#     pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32, device=device) if pos_weight else None
#     cls_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)(cls_out.squeeze(-1), labels.float())

#     target = wfall.permute(0, 2, 1)           # (B, n_freq, T) → (B, T, n_freq)
#     diff = (x_recon - target) ** 2            # (B, T, n_freq)
#     recon_loss = (diff.mean(dim=-1) * mask).sum() / mask.sum()

#     return alpha * cls_loss + beta * recon_loss, cls_loss, recon_loss

In [ ]:
class SinusoidalPE(nn.Module):
    def __init__(self, seq_len, embed_dim):
        super().__init__()
        pe = torch.zeros(seq_len, embed_dim)
        pos = torch.arange(seq_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, embed_dim, 2) * (-np.log(10000) / embed_dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe


class FRBMaskedAutoencoder(nn.Module):
    def __init__(self, seq_len, n_freq, embed_dim, mask_ratio=0.25):
        super().__init__()
        self.seq_len = seq_len
        self.n_freq = n_freq
        self.embed_dim = embed_dim
        self.mask_ratio = mask_ratio

        # --- Encoder ---
        self.enc_proj = nn.Linear(n_freq, embed_dim)
        # Fix 3: removed enc_embed / dec_embed redundant projections
        self.enc_pe = SinusoidalPE(seq_len + 1, embed_dim)
        self.enc_blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(embed_dim, nhead=2, dim_feedforward=128,
                                       batch_first=True) for _ in range(2)
        ])
        self.enc_norm = nn.LayerNorm(embed_dim)  # Fix 4: re-enable LayerNorm

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dec_pe = SinusoidalPE(seq_len + 1, embed_dim)
        self.dec_blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(embed_dim, nhead=2, dim_feedforward=128,
                                       batch_first=True) for _ in range(2)
        ])
        self.dec_norm = nn.LayerNorm(embed_dim)  # Fix 4: re-enable LayerNorm
        self.dec_proj = nn.Linear(embed_dim, n_freq)

        self.cls_head = nn.Linear(embed_dim, 1)

        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.mask_token, std=0.02)

    def mask_input(self, x):
        N, L, D = x.shape
        len_keep = int(L * (1 - self.mask_ratio))

        noise = torch.rand(N, L, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)

        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(x, dim=1,
                                index=ids_keep.unsqueeze(-1).expand(-1, -1, D))

        mask = torch.ones(N, L, device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)

        return x_masked, mask, ids_restore, ids_keep

    def encoder(self, x):
        x = self.enc_proj(x)                            # (B, T, E)
        x = x + self.enc_pe.pe[1:self.seq_len + 1] 
        x_vis, mask, ids_restore, ids_keep = self.mask_input(x)

        cls = (self.cls_token + self.enc_pe.pe[0]).expand(x_vis.size(0), -1, -1)
        x_vis = torch.cat([cls, x_vis], dim=1)          # (B, 1+n_keep, E)

        for block in self.enc_blocks:
            x_vis = block(x_vis)
        x_vis = self.enc_norm(x_vis)

        return x_vis, mask, ids_restore

    def decoder(self, x_enc, ids_restore):
        B = x_enc.size(0)
        T = ids_restore.size(1)
        n_keep = x_enc.size(1) - 1                      
        mask_tokens = self.mask_token.expand(B, T - n_keep, -1)

        x_no_cls = x_enc[:, 1:, :]                      # drop CLS
        x_full = torch.cat([x_no_cls, mask_tokens], dim=1)
        x_full = torch.gather(x_full, dim=1,
                              index=ids_restore.unsqueeze(-1).expand(-1, -1, self.embed_dim))

        x_full = x_full + self.dec_pe.pe[1:T + 1]
        cls = x_enc[:, :1, :] + self.dec_pe.pe[0]
        x_full = torch.cat([cls, x_full], dim=1)

        for block in self.dec_blocks:
            x_full = block(x_full)
        x_full = self.dec_norm(x_full)

        recon = self.dec_proj(x_full[:, 1:, :])
        return recon

    def forward(self, x):
        # x: (B, N_FREQ, T)
        x_t = x.permute(0, 2, 1)                       
        x_enc, mask, ids_restore = self.encoder(x_t)
        cls_out = self.cls_head(x_enc[:, 0, :])         
        recon = self.decoder(x_enc, ids_restore) 
        return recon, cls_out.squeeze(-1), mask


def compute_loss(cls_out, x_recon, mask, wfall, labels,
                 alpha=1.0, beta=1.0, pos_weight=None, device='cpu'):
    pw = torch.tensor([pos_weight], dtype=torch.float32, device=device) if pos_weight else None
    cls_loss = nn.BCEWithLogitsLoss(pos_weight=pw)(cls_out, labels.float())
    target = wfall.permute(0, 2, 1)                   
    diff = (x_recon - target) ** 2
    recon_loss = (diff * mask.unsqueeze(-1)).sum() / (mask.sum() * wfall.size(1))

    return alpha * cls_loss + beta * recon_loss, cls_loss, recon_loss

In [23]:
from sklearn.metrics import confusion_matrix


def train_one_epoch(model, loader, optimiser, device):
    model.train()
    total, correct = 0, 0
    running_loss = running_cls = running_recon = 0.0

    for wfall, labels in loader:
        wfall, labels = wfall.to(device), labels.to(device)

        x_recon, cls_out, mask = model(wfall)
        loss, cls_loss, recon_loss = compute_loss(cls_out, x_recon, mask, wfall, labels, device=device)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        running_loss  += loss.item()
        running_cls   += cls_loss.item()
        running_recon += recon_loss.item()

        preds = (torch.sigmoid(cls_out.squeeze(-1)) > 0.5).long()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    n = len(loader)
    print(f"  loss={running_loss/n:.4f}  cls={running_cls/n:.4f}  "
          f"recon={running_recon/n:.4f}  acc={correct/total:.3f}")


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total, correct = 0, 0
    
    # compute batch-wide confusion matrix here
    
    confusion_matrix_global = np.zeros((2, 2), dtype=int)  # [[TN, FP], [FN, TP]]
    
    for wfall, labels in loader:
        wfall, labels = wfall.to(device), labels.to(device)
        x_recon, cls_out, mask = model(wfall)
        preds = (torch.sigmoid(cls_out.squeeze(-1)) > 0.5).long()
        
        
        cf_mat = confusion_matrix(labels.cpu(), preds.cpu(), labels=[0, 1])
        confusion_matrix_global += cf_mat
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        
        

    print(f"  val acc={correct/total:.3f}")
    print(confusion_matrix_global)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = FRBMaskedAutoencoder(
    seq_len=TARGET_LENGTH,
    n_freq=256,
    embed_dim=64,
    mask_ratio=0.25,
).to(device)
optimiser = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

N_EPOCHS = 10
for epoch in range(N_EPOCHS):
    print(f"Epoch {epoch+1}/{N_EPOCHS}")
    train_one_epoch(model, train_loader, optimiser, device)
    evaluate(model, val_loader, device)

Epoch 1/10
  loss=1.3988  cls=0.4851  recon=0.9137  acc=0.780
  val acc=0.804
[[638  75]
 [103  92]]
Epoch 2/10
  loss=1.2813  cls=0.4037  recon=0.8775  acc=0.811
  val acc=0.809
[[699  14]
 [159  36]]
Epoch 3/10
  loss=1.2466  cls=0.3796  recon=0.8671  acc=0.826
  val acc=0.813
[[641  72]
 [ 98  97]]
Epoch 4/10
  loss=1.2089  cls=0.3505  recon=0.8584  acc=0.849
  val acc=0.814
[[671  42]
 [127  68]]
Epoch 5/10
  loss=1.1862  cls=0.3311  recon=0.8550  acc=0.861
  val acc=0.805
[[629  84]
 [ 93 102]]
Epoch 6/10
  loss=1.1726  cls=0.3150  recon=0.8576  acc=0.873
  val acc=0.805
[[649  64]
 [113  82]]
Epoch 7/10
  loss=1.1607  cls=0.3063  recon=0.8544  acc=0.883
  val acc=0.824
[[661  52]
 [108  87]]
Epoch 8/10
  loss=1.1392  cls=0.2876  recon=0.8516  acc=0.885
  val acc=0.830
[[657  56]
 [ 98  97]]
Epoch 9/10
  loss=1.1340  cls=0.2819  recon=0.8521  acc=0.886
  val acc=0.827
[[677  36]
 [121  74]]
Epoch 10/10
  loss=1.1025  cls=0.2522  recon=0.8503  acc=0.908
  val acc=0.828
[[658  55]
 